In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from PIL import Image
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel

In [ ]:
csv_path = r"C:\Users\abhis\OneDrive\Desktop\Mid-Fusion\2000_Fusion_Metadata.csv"
mri_folder = r"C:\Users\abhis\OneDrive\Desktop\Mid-Fusion\MF_MRI"

save_dir = "./saved_models"
os.makedirs(save_dir, exist_ok=True)

In [3]:
df = pd.read_csv(csv_path)

def clean_name(x):
    x = str(x).strip().lower()
    x = os.path.splitext(x)[0]
    return x

# Clean input_id (your actual column)
df["input_id"] = df["image_id"].apply(clean_name)

# Create image map
image_map = {}
for f in os.listdir(mri_folder):
    image_map[clean_name(f)] = os.path.join(mri_folder, f)

# ✅ CREATE NEW COLUMN (DO NOT overwrite ID)
df["image_path"] = df["input_id"].map(image_map)

# Drop unmatched rows
df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Total samples:", len(df))

Total samples: 2000


In [5]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["alzheimers"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.333, stratify=temp_df["alzheimers"], random_state=42
)

print(len(train_df), len(val_df), len(test_df))

1400 400 200


In [6]:
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
tokenizer.save_pretrained(save_dir + "/tokenizer")

('./2saved_models/tokenizer\\tokenizer_config.json',
 './2saved_models/tokenizer\\tokenizer.json')

In [7]:
class FusionDataset(Dataset):
    def __init__(self, df, tokenizer, transform=None):
        self.df = df
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ✅ MRI (use correct column)
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # ✅ TEXT
        encoding = self.tokenizer(
            str(row["text"]),
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        return {
            "image": img,
            "input_ids": encoding["input_ids"].squeeze(0),      # ✅ FIXED
            "attention_mask": encoding["attention_mask"].squeeze(0),

            # ✅ LABEL (must be named 'label')
            "label": torch.tensor(row["alzheimers"], dtype=torch.float)
        }

In [8]:
print(df[["text", "alzheimers"]].head())

                                                text  alzheimers
0  A 14-year-old girl, accompanied by his father,...           0
1  A 37-year-old Caucasian male was admitted in e...           1
2  This is regarding a 54-year-old Hispanic male ...           0
3  A 72-year old man, non-smoker, with a history ...           1
4  The 72 year old male patient was referred to o...           0


In [9]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_loader = DataLoader(FusionDataset(train_df, tokenizer, transform), batch_size=16, shuffle=True)
val_loader = DataLoader(FusionDataset(val_df, tokenizer, transform), batch_size=16)
test_loader = DataLoader(FusionDataset(test_df, tokenizer, transform), batch_size=16)

In [10]:
import torchvision.models as models

class MidFusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Linear(512, 256)

        self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.text_fc = nn.Linear(768, 256)

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, img, input_ids, mask):
        img_feat = self.cnn(img)
        img_feat = img_feat / 5  # ✅ scale balance

        text_out = self.bert(input_ids=input_ids, attention_mask=mask)
        text_feat = text_out.last_hidden_state[:, 0, :]
        text_feat = self.text_fc(text_feat)
        text_feat = text_feat / 5  # ✅ scale balance

        fused = torch.cat((img_feat, text_feat), dim=1)
        return self.classifier(fused)

In [11]:
class LateFusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Linear(512, 1)

        self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.text_fc = nn.Linear(768, 1)

        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, img, input_ids, mask):
        img_out = self.cnn(img)
        img_out = img_out / 5  # ✅ scale fix

        text_out = self.bert(input_ids=input_ids, attention_mask=mask)
        text_feat = text_out.last_hidden_state[:, 0, :]
        text_out = self.text_fc(text_feat)
        text_out = text_out / 5  # ✅ scale fix

        alpha = torch.sigmoid(self.alpha)  # ✅ constrain alpha

        return alpha * img_out + (1 - alpha) * text_out

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(outputs, labels):
    probs = torch.sigmoid(outputs).detach().cpu().numpy()
    preds = (probs > 0.5).astype(int)
    labels = labels.detach().cpu().numpy()

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.0

    return acc, precision, recall, f1, auc

In [13]:
def train_model(model, train_loader, val_loader, name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)  # ✅ updated LR
    criterion = nn.BCEWithLogitsLoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_precision": [], "val_precision": [],
        "train_recall": [], "val_recall": [],
        "train_f1": [], "val_f1": [],
        "train_auc": [], "val_auc": []
    }

    best_f1 = -1
    best_val_loss = float("inf")
    patience = 5
    counter = 0
    best_epoch = 0

    for epoch in range(1, 251):  # ✅ 1 to 250
        model.train()

        total_loss = 0
        all_outputs = []
        all_labels = []

        loop = tqdm(train_loader, desc=f"{name} Epoch {epoch}")

        for batch in loop:
            img = batch["image"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).unsqueeze(1)

            out = model(img, ids, mask)
            loss = criterion(out, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            all_outputs.append(out)
            all_labels.append(labels)

        all_outputs = torch.cat(all_outputs)
        all_labels = torch.cat(all_labels)

        train_acc, train_prec, train_rec, train_f1, train_auc = compute_metrics(all_outputs, all_labels)

        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc = evaluate_loss(
            model, val_loader, criterion, device
        )

        scheduler.step(val_f1)

        # Save history
        history["train_loss"].append(total_loss)
        history["val_loss"].append(val_loss)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        history["train_precision"].append(train_prec)
        history["val_precision"].append(val_prec)

        history["train_recall"].append(train_rec)
        history["val_recall"].append(val_rec)

        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)

        print(f"\n{name} Epoch {epoch}")
        print(f"Train → Loss:{total_loss:.4f} Acc:{train_acc:.4f} F1:{train_f1:.4f}")
        print(f"Val   → Loss:{val_loss:.4f} Acc:{val_acc:.4f} F1:{val_f1:.4f}")

        # ✅ TRACK BEST F1 CORRECTLY
        if not np.isnan(val_f1) and val_f1 > best_f1:
            best_f1 = val_f1

        # ✅ EARLY STOPPING (based on VAL LOSS)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            best_epoch = epoch

            torch.save(model.state_dict(), f"{save_dir}/{name}_best_weights.pth")
            torch.save(model, f"{save_dir}/{name}_best_full.pth")

        else:
            counter += 1

        if counter >= patience:
            print(f"\n Early stopping at epoch {epoch}")
            break

    # ✅ TRIM HISTORY TO BEST EPOCH
    for key in history:
        history[key] = history[key][:best_epoch]

    np.save(f"{save_dir}/{name}_history.npy", history)

    print(f"\n Best Epoch: {best_epoch}, Best F1: {best_f1:.4f}")

    return model, history

In [14]:
from tqdm import tqdm

def evaluate_loss(model, loader, criterion, device):
    model.eval()
    
    total_loss = 0
    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            img = batch["image"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).unsqueeze(1)

            out = model(img, ids, mask)
            loss = criterion(out, labels)

            total_loss += loss.item()

            all_outputs.append(out)
            all_labels.append(labels)

    all_outputs = torch.cat(all_outputs)
    all_labels = torch.cat(all_labels)

    acc, precision, recall, f1, auc = compute_metrics(all_outputs, all_labels)

    return total_loss, acc, precision, recall, f1, auc

In [15]:
batch = next(iter(train_loader))
print(batch.keys())

dict_keys(['image', 'input_ids', 'attention_mask', 'label'])


In [16]:
mid_model, mid_hist = train_model(MidFusionModel(), train_loader, val_loader, "mid_fusion")
late_model, late_hist = train_model(LateFusionModel(), train_loader, val_loader, "late_fusion")

c:\Users\abhis\OneDrive\Desktop\Alzheimer's Code\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\abhis\OneDrive\Desktop\Alzheimer's Code\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
mid_fusion Epoch 1: 100%|██████████| 88/88 [56:33<00:00, 38.56s/it] 



mid_fusion Epoch 1
Train → Loss:51.5529 Acc:0.7864 F1:0.7515
Val   → Loss:10.8929 Acc:0.8975 F1:0.8907


mid_fusion Epoch 2: 100%|██████████| 88/88 [1:00:28<00:00, 41.24s/it]


...


late_fusion Epoch 32
Train → Loss:2.3936 Acc:1.0000 F1:1.0000
Val   → Loss:4.4492 Acc:0.9325 F1:0.9313


late_fusion Epoch 33: 100%|██████████| 88/88 [33:11<00:00, 22.63s/it]



late_fusion Epoch 33
Train → Loss:2.7070 Acc:0.9993 F1:0.9993
Val   → Loss:4.5140 Acc:0.9400 F1:0.9388


late_fusion Epoch 34: 100%|██████████| 88/88 [33:19<00:00, 22.72s/it]



late_fusion Epoch 34
Train → Loss:2.2517 Acc:1.0000 F1:1.0000
Val   → Loss:4.4435 Acc:0.9375 F1:0.9364

 Early stopping at epoch 34

 Best Epoch: 29, Best F1: 0.9517


In [17]:
def evaluate(model, loader, name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    preds, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            img = batch["image"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            labels = batch["label"].numpy()

            out = torch.sigmoid(model(img, ids, mask)).cpu().numpy()
            preds.extend((out > 0.5).astype(int))
            labels_all.extend(labels)

    cm = confusion_matrix(labels_all, preds)
    print(f"\n{name} Confusion Matrix:\n", cm)

    print("\nClassification Report:\n", classification_report(labels_all, preds))

    np.save(f"{save_dir}/{name}_preds.npy", preds)
    np.save(f"{save_dir}/{name}_labels.npy", labels_all)

    return cm

In [18]:
cm_mid = evaluate(mid_model, test_loader, "mid_fusion")
cm_late = evaluate(late_model, test_loader, "late_fusion")


mid_fusion Confusion Matrix:
 [[95  5]
 [ 6 94]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.94      0.95      0.95       100
         1.0       0.95      0.94      0.94       100

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200


late_fusion Confusion Matrix:
 [[95  5]
 [ 7 93]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.93      0.95      0.94       100
         1.0       0.95      0.93      0.94       100

    accuracy                           0.94       200
   macro avg       0.94      0.94      0.94       200
weighted avg       0.94      0.94      0.94       200



In [19]:
def get_modality_contribution(model, loader):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    img_norms = []
    text_norms = []

    with torch.no_grad():
        for batch in loader:
            img = batch["image"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            img_feat = model.cnn(img)

            text_out = model.bert(input_ids=ids, attention_mask=mask)
            text_feat = model.text_fc(text_out.last_hidden_state[:, 0, :])

            img_norms.append(torch.norm(img_feat.detach(), dim=1).cpu().numpy())
            text_norms.append(torch.norm(text_feat.detach(), dim=1).cpu().numpy())

    img_norm = np.mean(np.concatenate(img_norms))
    text_norm = np.mean(np.concatenate(text_norms))

    total = img_norm + text_norm

    print("\n🔍 Mid-Fusion Modality Contribution:")
    print(f"MRI Contribution:  {img_norm / total:.4f}")
    print(f"Text Contribution: {text_norm / total:.4f}")

In [20]:
get_modality_contribution(mid_model, test_loader)


🔍 Mid-Fusion Modality Contribution:
MRI Contribution:  0.4634
Text Contribution: 0.5366
